# **Model**

# **XRF milk database**

In [1]:
# generating synthetic spectra with different noise levels
import pandas as pd
import numpy as np
import kennard_stone as ks

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

from synthetic import generate_synthetic_spectral_data

config = [
    {
        'nome': 'A',
        'n_amostras': 156,
        'picos': [250, 380, 550, 700, 850],  # 4 picos
        'amp_media': 1.0,
        'amp_std': 0.3,
        'larg_media': 15.0,
        'larg_std': 2.0,
        'ruido_std': 0.04
    },
    {
        'nome': 'B',
        'n_amostras': 146,
        'picos': [50, 250, 380, 550, 850],  # 3 picos (sem pico em 550)
        'amp_media': 1.4,
        'amp_std': 0.5,
        'larg_media': 15.0,
        'larg_std': 1.8,
        'ruido_std': 0.035
    }
]

data_complete = generate_synthetic_spectral_data(
    configuracao_classes=config,
    n_pontos=500,
    x_min=1,
    x_max=1000,
    seed=0
)

import pandas as pd
pd.options.plotting.backend = 'plotly' # setting plotly as the backend for pandas plotting 

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.iloc[:, 1:], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.iloc[:, 1:], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# Xcalclass_prep = Xcalclass.copy()
# Xpredclass_prep = Xpredclass.copy()

# preprocessings
import preprocessings as prepr  # preprocessing methods
Xcalclass_prep, mean_calclass  = prepr.mc(Xcalclass)
Xpredclass_prep = Xpredclass - mean_calclass

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-29 06:53:52,002 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-29 06:53:52,026 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [ ]:
from modeling import svm_optimized

svm_model = svm_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, aim='classification', kernel='rbf')
svm_model[0]


,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,SVC,1.0,1.0,1.0,"[[109, 0], [0, 102]]",1.0,1.0,1.0,"[[47, 0], [0, 44]]"


In [4]:
# calculando a covariancia entre cada variável espectral e a predição do modelo SVM
y_pred = svm_model[4]['SVC'].values # using the continuous predictions from SVM, extracting as 1D array
cov_scores = []
for col in Xcalclass_prep.columns:
    x_values = Xcalclass_prep[col].values
    covariance = np.cov(x_values, y_pred)[0, 1] # covariance between x and y
    cov_scores.append(covariance)
cov_scores_df = pd.DataFrame(cov_scores, index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df = np.abs(cov_scores_df)
cov_scores_df.plot()

In [5]:
X_sv = svm_model[3].support_vectors_            # shape (n_SV, n_features)
alpha_dual = svm_model[3].dual_coef_.ravel()    # shape (n_SV,)

# Calculando os coeficientes p usando os vetores de suporte e os multiplicadores de Lagrange
pvetor = pd.DataFrame({'energia' : Xcalclass.columns,
                       'importance': (X_sv.T) @ alpha_dual})
pvetor['importance'] = np.abs(pvetor['importance'])
pvetor['importance'].plot()

In [6]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('F1', 1.0, 100.0),
('background1', 100.0, 200.0),
('F2', 200.0, 300.0),
('background2', 300.0, 330.0),
('F3', 330.0, 430.0),
('background3', 430.0, 500.0),
('F4', 500.0, 600.0),
('background4', 600.0, 660.0),
('F5', 660.0, 750.0),
('background5', 750.0, 815.0),
('F6', 815.0, 890.0),
('background6', 890.0, 1000.0)
]

In [7]:
import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred
)

## Pvector and SHAP

In [8]:
# VIP scores por energia
pvector_df = pd.DataFrame({
    'energy': pvetor['energia'],
    'Pvector': pvetor['importance'].values
})
pvector_df = pvector_df.sort_values(by='Pvector', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in pvector_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
pvector_df['Zone'] = pvector_df['energy'].map(energy_to_zone_vip)
pvector_unique_df = pvector_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
pvector_unique_df = pvector_unique_df.sort_values(by='Pvector', ascending=False).reset_index(drop=True)
pvector_unique_df
shap_unique_df = pd.read_csv('shap_synthetic.csv', sep=';') # loading previously saved shap_unique_df

# permutation

In [9]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = svm_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = svm_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,49.048096192384776,0.003943,F1
1,699.6993987975952,0.001277,F5
2,847.8476953907816,0.000612,F6
3,379.3787575150301,0.000526,F3
4,549.5490981963928,0.000506,F4
5,249.248496993988,0.000443,F2
6,109.10821643286573,0.000010,background1
7,315.3146292585171,0.000010,background2
8,791.7915831663327,0.000010,background5
9,605.6052104208417,0.000008,background4


# **bagging - covariance**

In [17]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = pd.Series(y_pred) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.0001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graphv2(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',  # ou 'Covariance' se mudar a métrica
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 75 | Descartados: 21
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.0001

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartad

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,F1 > -0.71,F1 > -0.71,F1 > -0.71,F1 > -0.69
1,F1 > -0.69,F1 > -0.69,F1 <= 0.83,F1 > -0.71
2,F1 <= 0.83,F1 <= 0.83,F5 > -0.54,F1 <= 0.83
3,F5 > -0.54,F5 > -0.54,F1 > -0.69,F5 > -0.54
4,F5 <= 0.61,F5 <= 0.61,F5 <= 0.61,F5 <= 0.61
...,...,...,...,...
72,background5 > -0.10,F5 > 0.24,background6 <= -0.10,F5 > 0.24
73,background3 <= -0.08,background3 <= -0.09,F5 > 0.24,Class_A
74,background6 > 0.10,Class_A,Class_A,Class_B
75,Class_A,Class_B,Class_B,NaN


In [18]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df = lrc_summed_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,F1 > -0.71,12.562828,F1,-0.71,>
1,F5 > -0.54,9.489913,F5,-0.54,>
2,F2 > -0.45,2.418118,F2,-0.45,>
3,F4 > -0.24,2.360092,F4,-0.24,>
4,F6 > -0.25,2.218108,F6,-0.25,>
5,F3 > -0.43,2.128705,F3,-0.43,>
6,background4 > -0.09,0.241515,background4,-0.09,>
7,background3 > -0.08,0.189226,background3,-0.08,>
8,background1 > -0.10,0.174815,background1,-0.10,>
9,background6 <= 0.10,0.172316,background6,0.10,<=


# **Perturbation**

In [12]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = pd.Series(y_pred) # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=svm_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='mean', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graphv2(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',  # ou 'Covariance' se mudar a métrica
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 75 | Descartados: 21
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 73 | Descartados: 23
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 74 | Descartados: 22
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): classification
Modo de perturbação: mean
Fonte das estatísticas: full
Métrica: probability_shift
Total de folds:

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,F1 <= -0.69,F1 <= -0.69,F1 <= -0.69,F1 <= -0.69
1,F1 <= 0.22,F1 <= 0.22,F1 <= 0.22,F1 <= 0.22
2,F1 <= 0.83,F1 <= 0.83,F1 <= 0.83,F1 <= 0.83
3,F1 > -0.71,F1 > -0.71,F1 > -0.71,F1 > -0.71
4,F1 > -0.69,F1 > -0.69,F1 > -0.69,F1 > -0.69
...,...,...,...,...
74,background2 <= 0.07,background2 > -0.08,background2 <= -0.06,background2 <= -0.06
75,background2 <= 0.08,background2 <= 0.07,background2 <= 0.07,Class_A
76,background2 <= -0.06,Class_A,Class_A,Class_B
77,Class_A,Class_B,Class_B,NaN


In [13]:
all_results_pert[0]['pert_results_dict']['Bag_9']

,Predicate,Perturbation
0,F1 <= -0.69,0.231785
1,F1 <= 0.22,0.203058
2,F1 <= 0.83,0.155918
3,F1 > -0.71,0.096820
4,F1 > -0.69,0.069788
...,...,...
69,background2 > -0.06,0.000037
70,background2 > -0.08,0.000033
71,background2 <= 0.08,0.000026
72,background2 <= -0.06,0.000023


In [14]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df = lrc_summed_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,F1 <= -0.69,19.512936,F1,-0.69,<=
1,F5 <= -0.52,2.772248,F5,-0.52,<=
2,F6 <= -0.25,0.286932,F6,-0.25,<=
3,F3 > 0.14,0.204081,F3,0.14,>
4,F2 > 0.17,0.173143,F2,0.17,>
5,F4 <= -0.24,0.120774,F4,-0.24,<=
6,background6 > 0.10,0.004396,background6,0.10,>
7,background1 > 0.08,0.003565,background1,0.08,>
8,background5 > 0.08,0.003470,background5,0.08,>
9,background4 > 0.09,0.002983,background4,0.09,>


In [19]:
import numpy as np

max_len = max(
    len(pvector_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'SVM_pvector': pad_list(pvector_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation': pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,SVM_pvector,Shap,Permutation,LRC_perturbation,LRC_covariance
0,F1,F1,F1,F1,F1
1,F5,F5,F5,F5,F5
2,F6,F3,F6,F6,F2
3,F3,F4,F3,F3,F4
4,F4,F2,F4,F2,F6
5,F2,F6,F2,F4,F3
6,background2,background1,background1,background6,background4
7,background1,background2,background2,background1,background3
8,background4,background6,background5,background5,background1
9,background6,background3,background4,background4,background6


In [20]:
import rbo

rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        list_1 = features_importance[method_1].tolist()
        list_2 = features_importance[method_2].tolist()
        rbo_score = rbo.RankingSimilarity(list_1, list_2).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_2796\3888930707.py:12: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
1,SVM_pvector,Permutation,0.963578
7,Permutation,LRC_perturbation,0.946084
2,SVM_pvector,LRC_perturbation,0.944162
4,Shap,Permutation,0.878279
0,SVM_pvector,Shap,0.874447
5,Shap,LRC_perturbation,0.870148
6,Shap,LRC_covariance,0.866349
3,SVM_pvector,LRC_covariance,0.842546
9,LRC_perturbation,LRC_covariance,0.840624
8,Permutation,LRC_covariance,0.839413
